# Notebook 0.0 — Panorama de la Capa 1: Ingesta de datos y procesos en GCP
### Serie *Capa 1 del embudo AI-first · Ingesta de datos en GCP*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/noelserdna/capa1-ingesta-colab/blob/main/00_0_panorama_capa1_ingesta.ipynb)

> **Clase magistral · 90 min · Caso conductor: InnovaCo · Stack didáctico: Notion + Google Workspace + Telegram + Neon Postgres**

---

Este cuaderno es el **mapa de toda la Capa 1**. Va **antes** del Notebook 0 (*Fundamentos y setup*): aquí ves el *bosque* —las 7 tecnologías de ingesta de GCP, cuándo usar cada una y las trampas clásicas— y en los siguientes cuadernos bajas al *árbol* y lo ejecutas.

**Para quién:** developers senior que ya conocen los fundamentos de GCP e IAM y quieren diseñar estrategias de ingesta AI-first.

**Cómo leer esta guía:**
- El texto explica los conceptos; los **bloques de código** (YAML, `gcloud`, prompts, Python) son **material de referencia para copiar**, no para ejecutar aquí —su despliegue real ocurre con tu proyecto GCP ya configurado (Notebook 0 en adelante).
- Hay algunas **celdas Python ejecutables** (mapa mental, recomendador de tecnología, plantilla de prompts) que sí puedes correr sin ningún setup: son autocontenidas.

**Objetivos de aprendizaje** — al terminar sabrás:
1. Distinguir cuándo usar **Application Integration**, **Workflows**, **Pub/Sub**, **Eventarc**, **Datastream**, **Storage Transfer Service** y **Cortex Framework**.
2. Diseñar una estrategia de ingesta híbrida (batch + streaming + event-driven) para una pyme con stack SaaS.
3. Identificar las **trampas clásicas** de cada tecnología antes de pisarlas.
4. Escribir el primer flujo de ingesta de extremo a extremo… pidiéndoselo a un LLM con un buen prompt.


## Cronograma de la clase

| Bloque | Tema | Min |
|---|---|---|
| 0 | Apertura y mapa mental de la capa 1 | 5 |
| 1 | Application Integration | 15 |
| 2 | Workflows | 12 |
| 3 | Pub/Sub | 13 |
| 4 | Eventarc | 10 |
| 5 | Datastream | 10 |
| 6 | Cloud Storage Transfer Service | 8 |
| 7 | Cortex Framework | 12 |
| 8 | Cierre, decisión arquitectónica y Q&A | 5 |

> **Prerrequisitos:** Fundamentos GCP · IAM y service accounts · concepto del embudo AI-first.


---
## Bloque 0 — Apertura

### La pregunta que abre la clase

> *"Tu cliente tiene HubSpot, Holded, Slack, Asana, Drive y un Postgres viejo en una VPS. Te pide que toda esa información llegue a un sitio común para que una IA razone sobre ella. ¿Por dónde empiezas?"*

Esta es exactamente la situación de **InnovaCo** y es donde vive la **Capa 1 — Ingesta**.

### La regla de oro

> **Si no entra al embudo, la IA no puede razonar sobre ello.**

La cobertura de ingesta es **estratégica, no técnica**. No se trata de "qué API se conecta más fácil" sino de **qué información de la empresa necesitamos para los outputs que queremos generar**.

### Mapa mental — los cuatro modos de ingesta

| Modo | Cuándo | Tecnología principal |
|---|---|---|
| **Batch periódico** | Datos que no cambian al minuto: CRM, ERP, facturación | Application Integration, Storage Transfer |
| **Streaming** | Datos que cambian al segundo: pedidos, eventos web, logs | Pub/Sub + Dataflow |
| **CDC (Change Data Capture)** | Espejar una BD relacional sin tocar la app | Datastream |
| **Event-driven** | Reaccionar cuando algo pasa: archivo subido, ticket creado | Eventarc |

La mayoría de pymes empiezan **solo con batch**, suben a **event-driven** cuando descubren casos donde "el viernes" es tarde, y a **streaming/CDC** cuando ya tienen volumen.

### La trampa que el embudo evita

Todas las herramientas se solapan parcialmente. La trampa clásica es **usar la misma para todo**. La estrategia AI-first es elegir cada una para lo que está optimizada y **orquestar el conjunto**.


In [ ]:
# Mapa mental de los 4 modos de ingesta — celda autocontenida, ejecútala sin setup.
modos = {
    "Batch periódico": {
        "cuando": "Datos que no cambian al minuto: CRM, ERP, facturación",
        "tecnologia": "Application Integration, Storage Transfer",
    },
    "Streaming": {
        "cuando": "Datos que cambian al segundo: pedidos, eventos web, logs",
        "tecnologia": "Pub/Sub + Dataflow",
    },
    "CDC (Change Data Capture)": {
        "cuando": "Espejar una BD relacional sin tocar la app",
        "tecnologia": "Datastream",
    },
    "Event-driven": {
        "cuando": "Reaccionar cuando algo pasa: archivo subido, ticket creado",
        "tecnologia": "Eventarc",
    },
}

for nombre, info in modos.items():
    print(f"\n### {nombre}")
    print(f"  Cuándo     : {info['cuando']}")
    print(f"  Tecnología : {info['tecnologia']}")


### El modo de trabajo de esta clase: IA + gcloud CLI

En esta clase **no escribimos el código a mano**. El patrón —y el que tus alumnos deben adoptar— es:

1. **Pensar la arquitectura** (esto es lo que aporta el dev senior; la IA no lo reemplaza).
2. **Escribir un prompt preciso** describiendo qué queremos, en qué contexto, con qué constraints.
3. **Pedirle al LLM** (`gemini`, `claude`, ChatGPT…) que genere el YAML, el comando, la función o el script.
4. **Revisar**: ¿hay idempotencia? ¿retries? ¿secretos hardcodeados? ¿DLQ?
5. **Desplegar con `gcloud`**.

Esto es **AI-augmented engineering**. Los **prompts forman parte del código** —se versionan en Git como cualquier otra cosa. Para cada bloque verás el mismo recorrido:

> *Ejemplo concreto en InnovaCo* → *Prompt para el LLM* → *Lo que el LLM debería generar* → *Comando `gcloud` para desplegar*

### Anatomía de un prompt bueno

Un prompt útil para infraestructura GCP **siempre** incluye:

- **Rol / contexto**: "eres un ingeniero senior GCP, trabajamos en el proyecto X".
- **Objetivo concreto**: qué tiene que producir (YAML, gcloud command, Python).
- **Stack y restricciones**: región, proyecto, service account, naming, idempotencia, retries.
- **Esquema de los datos** (si aplica): nombres de tablas/campos.
- **Trampas a evitar**: "sin secretos en claro, usar Secret Manager".
- **Formato de salida**: "devuélveme solo el YAML, sin explicación".


### Stack didáctico — qué necesitas instalado

Para ejecutar los ejemplos en clase sin fricción usamos un stack accesible y gratis:

| Pieza | Producto | Coste | Setup |
|---|---|---|---|
| CRM / tareas / docs internos | **Notion** | Gratis uso personal | API token desde Settings → Connections |
| Correo + Drive + Forms + Sheets + Calendar | **Google Workspace** | Free / edu | Service account con Domain-Wide Delegation |
| Notificaciones internas | **Telegram bot** | Gratis | `@BotFather`, 1 minuto |
| BD relacional para CDC | **Neon Postgres** | Free tier | Cuenta neon.tech, 2 minutos |
| Cloud | **Google Cloud** | Free tier | Proyecto `innovaco-prod` |

> Guía paso a paso para preparar el stack: **Apéndice E** al final.

### Mapeo InnovaCo real ↔ stack didáctico

| Concepto del caso | Stack real | Stack didáctico (clase) |
|---|---|---|
| Pipeline comercial | HubSpot CRM | Notion DB "Oportunidades" |
| Facturación y clientes | Holded ERP | Google Sheets "Clientes" + "Facturas" |
| Captura de leads | Formulario web | Google Forms → Sheets |
| Comunicación interna | Slack | Telegram bot |
| Gestión de proyectos | Asana | Notion DB "Proyectos" (kanban) |
| Documentos firmados | Drive corporativo | Google Drive (idéntico) |
| BD operacional para CDC | Cloud SQL Postgres | Neon Postgres serverless |
| Datos de cliente externo | SFTP del cliente | Carpeta Drive compartida |

> **El embudo arquitectónico es idéntico** — solo cambian las fuentes y los destinos. La arquitectura es independiente de las herramientas operacionales.


---
## Bloque 1 — Application Integration

### Concepto en 30 segundos

Plataforma **low-code** de orquestación con +100 conectores SaaS pre-construidos (HubSpot, Salesforce, Slack, Asana, Shopify, NetSuite, Workday, Jira…). Diseñas el flujo en un canvas visual, lo despliegas, y Google se ocupa de autenticación, paginación, retries y observabilidad. Es el sucesor de Apigee Integration; compite con Workato, Zapier corporativo, MuleSoft, Boomi.

### Anatomía

- **Trigger**: qué inicia el flujo (cron, webhook, API, evento Pub/Sub, calendario).
- **Tasks**: pasos secuenciales o paralelos (conector SaaS, transformar datos, escribir en BigQuery, sub-integración, lógica condicional).
- **Edges**: conexiones entre tasks, con condiciones opcionales.
- **Connection** (Integration Connectors): conexión reutilizable a un SaaS, con credenciales gestionadas.
- **Auth profile**: las credenciales (OAuth, API key, basic auth).

### Cuándo usarlo
- **SaaS con conector pre-hecho** → ahorras semanas.
- **Flujos low/medium complexity** (5-30 pasos, lógica visualizable).
- **Equipos mixtos** (devs + ops + business): el canvas baja la barrera.
- **Cuando quieres que los datos no salgan de GCP**: el conector vive en tu proyecto.

### Cuándo NO usarlo
- **Lógica muy compleja** (50+ pasos) → Workflows o código.
- **Procesamiento pesado** (GB en transformación) → Dataflow.
- **SaaS sin conector** → conexión custom (REST), pierde la ventaja low-code.
- **Tiempo crítico submilisegundo**.

### Ejemplo concreto en InnovaCo
**Caso**: ingesta diaria de la base Notion "Oportunidades" a BigQuery.

**Flujo**: cron `0 3 * * *` → REST task a la API de Notion (con filtro `last_edited_time`) → Data Mapping al esquema canónico → connector BigQuery con **MERGE** por `oportunidad_id` → en paralelo, si hubo errores, publica en Pub/Sub. **Resultado**: dataset `innovaco_raw_notion` actualizado cada mañana.


### Prompt para el LLM

```text
Eres un ingeniero senior de Google Cloud Platform.

CONTEXTO
Proyecto: innovaco-prod (region europe-west1).
Stack origen: base de datos Notion "Oportunidades", database_id = "DB_ID_NOTION_AQUI".
  - Token de integración Notion en Secret Manager bajo "notion-api-token".
  - Endpoint API: https://api.notion.com/v1/databases/{DB_ID}/query (POST).
  - Header obligatorio: "Notion-Version: 2022-06-28".
Stack destino: BigQuery, dataset innovaco_raw_notion, tabla oportunidades.
Service account: integration-notion-sa@innovaco-prod.iam.gserviceaccount.com.

OBJETIVO
Genera la definición de una integración en Application Integration que:
1. Se dispare cada día a las 03:00 hora Madrid (Europe/Madrid).
2. Llame a la API de Notion (REST task) con filtro last_edited_time > last_run_at,
   page_size 100, manejando paginación con next_cursor hasta has_more = false.
3. Transforme cada page al esquema canónico:
   - oportunidad_id <- id
   - oportunidad_nombre <- properties.Nombre.title[0].plain_text
   - empresa <- properties.Empresa.rich_text[0].plain_text
   - importe <- properties.Importe.number  (NUMERIC)
   - probabilidad_cierre <- properties.Probabilidad.number  (NUMERIC)
   - estado <- properties.Estado.select.name
   - fecha_modificacion <- last_edited_time  (TIMESTAMP)
4. Inserte en BigQuery con MERGE por oportunidad_id (idempotencia).
5. En paralelo, si hubo errores, publique en Pub/Sub "innovaco.ingesta.errors".
6. Actualice "last_run_at" al timestamp de inicio.

RESTRICCIONES
- Token Notion via Secret Manager, nunca hardcoded.
- Si Notion devuelve 429 (límite 3 req/s), retry con backoff exponencial (3 intentos, base 5s).
- Logging básico (no loguear el contenido completo de las páginas — PII de clientes).

FORMATO
Devuélveme la definición en YAML legible (vista lógica simplificada). Sin explicación.
```


### Lo que el LLM debería generar

```yaml
# Vista lógica de la integración (la UI es visual; este YAML es la intención que pediste)
integration:
  name: ingesta-notion-oportunidades-diaria
  triggers:
    - type: cron
      schedule: "0 3 * * *"
      timezone: "Europe/Madrid"
  variables:
    - name: last_run_at
      type: timestamp
      default: "1970-01-01T00:00:00Z"
    - name: notion_db_id
      type: string
      default: "DB_ID_NOTION_AQUI"
  tasks:
    - id: read_notion_pages
      type: rest
      method: POST
      url: "https://api.notion.com/v1/databases/{{ vars.notion_db_id }}/query"
      headers:
        Authorization: "Bearer {{ secret('notion-api-token') }}"
        Notion-Version: "2022-06-28"
        Content-Type: "application/json"
      body:
        filter:
          timestamp: "last_edited_time"
          last_edited_time:
            after: "{{ vars.last_run_at }}"
        page_size: 100
      pagination:
        next_token_path: "next_cursor"
        has_more_path: "has_more"
      retry:
        on_status: [429, 500, 502, 503]
        max_attempts: 3
        backoff: exponential
        base_seconds: 5
    - id: transform
      type: data_mapping
      input: read_notion_pages.results
      mapping:
        oportunidad_id: input.id
        oportunidad_nombre: input.properties.Nombre.title[0].plain_text
        empresa: input.properties.Empresa.rich_text[0].plain_text
        importe: input.properties.Importe.number
        probabilidad_cierre: input.properties.Probabilidad.number
        estado: input.properties.Estado.select.name
        fecha_modificacion: input.last_edited_time
    - id: write_bigquery
      type: connector
      connector: bigquery
      operation: merge
      dataset: innovaco_raw_notion
      table: oportunidades
      merge_key: oportunidad_id
      data: transform.output
    - id: update_last_run
      type: variable_set
      variable: last_run_at
      value: "{{ trigger.execution_started_at }}"
    - id: publish_errors
      type: connector
      connector: pubsub
      operation: publish
      topic: "innovaco.ingesta.errors"
      condition: "{{ tasks.read_notion_pages.error != null or tasks.write_bigquery.error != null }}"
      message:
        fuente: "notion"
        error: "{{ tasks.read_notion_pages.error or tasks.write_bigquery.error }}"
```


### Comando gcloud

```bash
# Listar integraciones del proyecto
gcloud integration integrations list --location=europe-west1

# Ver detalles de una ejecución concreta
gcloud integration executions describe EXEC_ID \
  --integration=ingesta-notion-oportunidades-diaria \
  --location=europe-west1

# Disparar una ejecución manualmente
gcloud integration integrations execute ingesta-notion-oportunidades-diaria \
  --location=europe-west1 \
  --input-parameters='{"force_full_refresh": true}'
```

### Trampas comunes
1. **No probar antes de publicar**: los conectores fallan silenciosamente si las credenciales caducan.
2. **Sin idempotencia**: un reintento puede duplicar registros. Usar `merge` por clave, no `insert`.
3. **Logging excesivo con PII**: si activas "log all task data" llenas Cloud Logging de datos sensibles.
4. **Hardcodear secretos** en variables del flujo: usa Secret Manager.
5. **Integración monolítica de 80 pasos**: divide en sub-integraciones. Mejor diez de 10 que una de 80.

### Ejercicio mental
> *"Cada nueva fila del Google Sheet 'Facturas' debe sincronizarse como nota en la página Notion 'Oportunidades' correspondiente, en cuanto se añada. Diseña el flujo en 5 tasks. ¿Qué trigger usas? ¿Cómo resuelves el matching factura ↔ oportunidad?"*

*Pista*: "en cuanto se añada" implica tiempo real → frontera con event-driven (Bloque 4).


---
## Bloque 2 — Workflows

### Concepto en 30 segundos
Orquestador **serverless** basado en YAML/JSON donde describes pasos, condicionales, paralelismos y retries en lenguaje declarativo. Cada paso llama a un endpoint HTTP, un servicio GCP o un conector. **Cero infraestructura**, pago por ejecución. Compite con Step Functions (AWS), Logic Apps (Azure), Temporal.

### Workflows vs. Application Integration

| Eje | Workflows | Application Integration |
|---|---|---|
| **Definición** | YAML/JSON declarativo, versionable | Canvas visual + YAML interno |
| **Curva** | Devs lo aceptan rápido | Más amigable para no-devs |
| **Lógica compleja** | Mejor (condicionales, loops, paralelismo nativos) | Posible pero verbose |
| **Versionado Git** | Natural | Manual (export YAML) |
| **Coste** | Por step ejecutado | Por ejecución |
| **Prefiérelo cuando** | Equipo técnico, GitOps | Equipos mixtos, prototipado |

### Cuándo usarlo
- **Lógica condicional fina o loops**.
- **Orquestación entre servicios GCP** (Functions + Run + BigQuery + Vertex AI).
- **Equipo técnico** que valora código-como-configuración.
- **Pipelines humano-máquina** con aprobaciones / callbacks.

### Cuándo NO usarlo
- **Flujos triviales** de 2-3 pasos.
- **Procesamiento masivo** (eso es Dataflow).
- **Streaming continuo**.

### Ejemplo concreto en InnovaCo
**Caso**: onboarding automático cuando se firma un contrato (nivel N2 de autonomía). Si el valor < 5000 € → solo notifica por Telegram. Si ≥ 5000 € → en paralelo crea page en Notion "Proyectos", page en Notion "Clientes", carpeta en Drive y fila en Sheets; al terminar, avisa por Telegram.


### Prompt para el LLM

```text
Eres un ingeniero senior de GCP especializado en orquestación serverless.

CONTEXTO
Proyecto: innovaco-prod (region europe-west1).
Service account del workflow: onboarding-sa@innovaco-prod.iam.gserviceaccount.com.
Stack didáctico:
  - Notion API: token en Secret Manager "notion-api-token". Database IDs en
    "notion-db-clientes-id" y "notion-db-proyectos-id". API base https://api.notion.com/v1.
  - Google Sheets: spreadsheet "clientes-maestro", id en "sheets-clientes-id". Hoja "Clientes".
  - Google Drive: carpeta raíz "Clientes" con id en "drive-carpeta-clientes-id".
  - Telegram: bot token en "telegram-bot-token", chat_id en "telegram-chat-id".

DATOS DE ENTRADA
Parámetro contrato_id (string).
Tabla origen: innovaco_curated_contratos.contratos_firmados
  (contrato_id, cliente_id, cliente_nombre, valor NUMERIC, tipo_proyecto).

OBJETIVO
Genera un workflow YAML que:
1. Cargue el contrato desde BigQuery por contrato_id.
2. Si valor < 5000 EUR: solo Telegram al equipo. Termina.
3. Si valor >= 5000 EUR: ejecute en PARALELO 4 ramas:
   a) Notion — page nueva en "Proyectos" (Nombre, Estado "Kickoff", Cliente, Valor).
   b) Notion — page nueva en "Clientes" (Nombre, cliente_id, Tipo "Activo").
   c) Drive — sub-carpeta dentro de "Clientes" con nombre cliente_nombre.
   d) Sheets — añadir fila [cliente_id, cliente_nombre, tipo_proyecto, valor, fecha_alta_hoy].
4. Al terminar, Telegram: "Onboarding completo: <cliente_nombre>...".

RESTRICCIONES
- Idempotencia: GET antes de crear; si cliente_id ya existe, saltar.
- Cada llamada HTTP con retry exponencial (3 intentos, base 2s).
- Sin secretos en claro: sys.get_env() referenciado a Secret Manager.
- Logging de cada paso con cliente_id como label.

FORMATO
Devuélveme solo el YAML del workflow, listo para "gcloud workflows deploy". Sin explicación.
```


### Lo que el LLM debería generar

```yaml
# workflow: onboarding-cliente-firmado.yaml  (stack didáctico)
main:
  params: [contrato_id]
  steps:
    - cargar_secretos:
        parallel:
          shared: [secretos]
          branches:
            - notion_token:
                steps:
                  - get_notion:
                      call: googleapis.secretmanager.v1.projects.secrets.versions.access
                      args:
                        name: projects/innovaco-prod/secrets/notion-api-token/versions/latest
                      result: notion_token_secret
            - telegram_token:
                steps:
                  - get_tg:
                      call: googleapis.secretmanager.v1.projects.secrets.versions.access
                      args:
                        name: projects/innovaco-prod/secrets/telegram-bot-token/versions/latest
                      result: tg_token_secret
            - telegram_chat:
                steps:
                  - get_chat:
                      call: googleapis.secretmanager.v1.projects.secrets.versions.access
                      args:
                        name: projects/innovaco-prod/secrets/telegram-chat-id/versions/latest
                      result: tg_chat_secret

    - cargar_contrato:
        call: googleapis.bigquery.v2.jobs.query
        args:
          projectId: innovaco-prod
          body:
            query: |
              SELECT cliente_id, cliente_nombre, valor, tipo_proyecto
              FROM innovaco_curated_contratos.contratos_firmados
              WHERE contrato_id = '${contrato_id}'
        result: contrato

    - validar:
        switch:
          - condition: ${contrato.valor < 5000}
            next: solo_notificar
          - condition: ${contrato.valor >= 5000}
            next: onboarding_completo

    - solo_notificar:
        call: http.post
        args:
          url: ${"https://api.telegram.org/bot" + tg_token_secret + "/sendMessage"}
          body:
            chat_id: ${tg_chat_secret}
            text: ${"Contrato pequeño firmado: " + contrato_id + " - " + contrato.cliente_nombre + " (" + string(contrato.valor) + "EUR)"}
        next: end

    - onboarding_completo:
        parallel:
          shared: [contrato, notion_token_secret]
          branches:
            - crear_proyecto_notion:
                steps:
                  - post_proyecto:
                      call: http.post
                      args:
                        url: "https://api.notion.com/v1/pages"
                        headers:
                          Authorization: ${"Bearer " + notion_token_secret}
                          Notion-Version: "2022-06-28"
                          Content-Type: "application/json"
                        body:
                          parent:
                            database_id: ${sys.get_env("NOTION_DB_PROYECTOS_ID")}
                          properties:
                            Nombre: { title: [{ text: { content: ${contrato.cliente_nombre + " - " + contrato.tipo_proyecto} } }] }
                            Estado: { select: { name: "Kickoff" } }
                            cliente_id: { rich_text: [{ text: { content: ${contrato.cliente_id} } }] }
            - crear_cliente_notion:
                steps:
                  - post_cliente_notion:
                      call: http.post
                      args:
                        url: "https://api.notion.com/v1/pages"
                        headers:
                          Authorization: ${"Bearer " + notion_token_secret}
                          Notion-Version: "2022-06-28"
                          Content-Type: "application/json"
                        body:
                          parent:
                            database_id: ${sys.get_env("NOTION_DB_CLIENTES_ID")}
                          properties:
                            Nombre: { title: [{ text: { content: ${contrato.cliente_nombre} } }] }
                            cliente_id: { rich_text: [{ text: { content: ${contrato.cliente_id} } }] }
                            Tipo: { select: { name: "Activo" } }
            - crear_carpeta_drive:
                steps:
                  - llamada_drive:
                      call: googleapis.drive.v3.files.create
                      args:
                        body:
                          name: ${contrato.cliente_nombre}
                          mimeType: "application/vnd.google-apps.folder"
                          parents: [ ${sys.get_env("DRIVE_CARPETA_CLIENTES_ID")} ]
            - registrar_sheets:
                steps:
                  - append_sheets:
                      call: googleapis.sheets.v4.spreadsheets.values.append
                      args:
                        spreadsheetId: ${sys.get_env("SHEETS_CLIENTES_ID")}
                        range: "Clientes!A:E"
                        valueInputOption: "USER_ENTERED"
                        body:
                          values:
                            - - ${contrato.cliente_id}
                              - ${contrato.cliente_nombre}
                              - ${contrato.tipo_proyecto}
                              - ${contrato.valor}
                              - ${time.format(sys.now())}

    - resumen_a_telegram:
        call: http.post
        args:
          url: ${"https://api.telegram.org/bot" + tg_token_secret + "/sendMessage"}
          body:
            chat_id: ${tg_chat_secret}
            text: ${"Onboarding completo: " + contrato.cliente_nombre + ". Notion + Drive + Sheets actualizados."}
        next: end

    - end:
        return: "OK"
```


### Comando gcloud

```bash
# Desplegar
gcloud workflows deploy onboarding-cliente-firmado \
  --source=onboarding-cliente-firmado.yaml \
  --location=europe-west1 \
  --service-account=onboarding-sa@innovaco-prod.iam.gserviceaccount.com

# Ejecutar manualmente
gcloud workflows execute onboarding-cliente-firmado \
  --location=europe-west1 \
  --data='{"contrato_id":"CTR-2026-0042"}'

# Ver ejecuciones
gcloud workflows executions list \
  --workflow=onboarding-cliente-firmado \
  --location=europe-west1
```

### Trampas comunes
1. **Olvidar permisos de la service account**: el workflow corre con su SA.
2. **No definir retries** para llamadas HTTP: las redes fallan.
3. **No manejar timeouts**: el workflow dura hasta 1 año, pero los steps HTTP tienen su propio timeout.
4. **Loops sin límite**: un `for` mal escrito se ejecuta miles de veces.
5. **No usar `parallel`** cuando los pasos son independientes: paralelizar baja la duración total.

### Ejercicio mental
> *"Adapta el workflow para que si el valor ≥ 50.000 € requiera aprobación explícita de dirección antes de crear recursos. ¿Cómo implementas 'esperar aprobación humana' en un workflow serverless?"*

*Pista*: el patrón **callback** — el workflow se pausa esperando una llamada HTTP a un endpoint generado. Es el **HITL** del nivel N2.


---
## Bloque 3 — Pub/Sub

### Concepto en 30 segundos
Cola de mensajes **distribuida, escalable, durable, totalmente managed**. Publicas en un **topic**, los suscriptores reciben desde una **subscription**. Entrega "at-least-once" (dedup posible para exactly-once). Es el **backbone de eventos** en GCP, equivalente a Kafka pero sin gestionar clusters.

### Anatomía
- **Topic**: el "canal" (`innovaco.leads.nuevos`).
- **Message**: blob JSON (hasta 10 MB) con atributos clave/valor.
- **Subscription**: consumidor **pull** (pide) o **push** (Pub/Sub empuja a un endpoint HTTP).
- **Schema** (opcional): valida con Avro o Protobuf.
- **DLQ**: topic donde van los mensajes que fallan tras N intentos.
- **Ordering keys** / **Filtering**.

### Cuándo usarlo
- **Eventos asíncronos entre servicios** ("se ha creado un cliente").
- **Buffer** entre productor y consumidor lento.
- **Fan-out**: un evento, varias suscripciones en paralelo.
- **Pipelines**: Pub/Sub → Dataflow → BigQuery.

### Cuándo NO usarlo
- **Request/response síncrono** (eso es Cloud Run + HTTP).
- **Estado/transacciones** (no es una BD).
- **Orden estricto a gran escala** (ordering keys limitan throughput).

### Ejemplo concreto en InnovaCo
**Caso**: un lead llega por **Google Form**. Tres cosas independientes deben pasar: (1) añadirlo a Notion "Oportunidades", (2) notificar al equipo por Telegram, (3) disparar el agente Gemini de calificación. **Sin Pub/Sub** el Apps Script es monolítico: si Notion falla, no se notifica; si el agente tarda, el form se cuelga. **Con fan-out**, las tres reaccionan en paralelo y desacopladas.


### Prompt para el LLM

```text
Eres un ingeniero senior de GCP. Construye el sistema de fan-out de eventos de leads en
Pub/Sub para InnovaCo (stack didáctico).

CONTEXTO
Proyecto: innovaco-prod (region europe-west1).
Tres consumidores independientes por cada lead nuevo:
  - lead-a-notion (Cloud Function que crea page en Notion "Oportunidades").
  - lead-a-telegram (Cloud Function que envía mensaje al chat de equipo).
  - lead-a-agente-calificacion (Cloud Run con agente Vertex AI Gemini).
Secretos: notion-api-token, notion-db-oportunidades-id, telegram-bot-token, telegram-chat-id.

OBJETIVO
1. Esquema Avro "lead-schema" (nombre, empresa, email, interes string; origen enum
   web/evento/referral/otro; fecha timestamp-millis).
2. Topic "innovaco.leads.nuevos" con ese esquema, encoding JSON.
3. Topic DLQ "innovaco.leads.nuevos.dlq".
4. Tres subscriptions push:
   - lead-a-notion -> Cloud Function "create-notion-lead", max delivery 5, DLQ.
   - lead-a-telegram -> Cloud Function "notify-telegram", max delivery 3, DLQ.
   - lead-a-agente-calificacion -> Cloud Run, FILTRO valor_estimado="alto" OR "medio", max 5, DLQ.
5. Apps Script (Sheets) onFormSubmit que publica en Pub/Sub vía REST, con atributo
   valor_estimado por heurística (interes contiene "consultoria estrategica"->alto,
   "formacion"->medio, resto->bajo).
6. Cloud Function Python "create-notion-lead": recibe push, decodifica base64, parsea JSON,
   POST a Notion creando la page.

RESTRICCIONES
- Idempotencia: dedup por email (Notion no tiene unique; primero GET con filtro).
- Service account "pubsub-push-sa" con roles/run.invoker en cada destino.
- Secretos via Secret Manager.

FORMATO
Bloques separados: (1) gcloud, (2) Apps Script (.gs), (3) Cloud Function Python.
```


### Lo que el LLM debería generar — Apps Script (productor)

```javascript
// Apps Script en el Sheet de respuestas del Google Form. Trigger: "Al enviar formulario".
function onFormSubmit(e) {
  const values = e.values;
  const lead = {
    nombre: values[1], empresa: values[2], email: values[3],
    interes: values[4], origen: values[5] || "web",
    fecha: new Date().toISOString()
  };

  const interesLower = (lead.interes || "").toLowerCase();
  let valor_estimado = "bajo";
  if (interesLower.includes("consultoría") || interesLower.includes("estratégica")) valor_estimado = "alto";
  else if (interesLower.includes("formación")) valor_estimado = "medio";

  const payload = {
    messages: [{
      data: Utilities.base64Encode(JSON.stringify(lead)),
      attributes: { valor_estimado: valor_estimado, origen: lead.origen }
    }]
  };

  UrlFetchApp.fetch(
    "https://pubsub.googleapis.com/v1/projects/innovaco-prod/topics/innovaco.leads.nuevos:publish",
    {
      method: "post",
      contentType: "application/json",
      payload: JSON.stringify(payload),
      headers: { Authorization: "Bearer " + ScriptApp.getOAuthToken() }
    }
  );
}
```


### Las tres subscriptions del mismo topic

```bash
# 1) Crear en Notion (Cloud Function)
gcloud pubsub subscriptions create lead-a-notion \
  --topic=innovaco.leads.nuevos \
  --push-endpoint=https://europe-west1-innovaco-prod.cloudfunctions.net/create-notion-lead \
  --dead-letter-topic=innovaco.leads.nuevos.dlq \
  --max-delivery-attempts=5

# 2) Notificar Telegram (Cloud Function)
gcloud pubsub subscriptions create lead-a-telegram \
  --topic=innovaco.leads.nuevos \
  --push-endpoint=https://europe-west1-innovaco-prod.cloudfunctions.net/notify-telegram \
  --dead-letter-topic=innovaco.leads.nuevos.dlq \
  --max-delivery-attempts=3

# 3) Agente IA solo para leads de valor alto o medio
gcloud pubsub subscriptions create lead-a-agente-calificacion \
  --topic=innovaco.leads.nuevos \
  --push-endpoint=https://agent-calificacion-xxx.run.app \
  --filter='attributes.valor_estimado="alto" OR attributes.valor_estimado="medio"' \
  --dead-letter-topic=innovaco.leads.nuevos.dlq \
  --max-delivery-attempts=5
```

Si Telegram está caído, Notion y el agente siguen. Añadir una cuarta acción mañana = una subscription nueva, sin tocar las anteriores. **Esto es desacoplamiento puro.**


### El consumidor — Cloud Function Notion (Python)

```python
# main.py — Cloud Function "create-notion-lead": recibe push de Pub/Sub y crea page en Notion.
import base64, json, os, requests
from google.cloud import secretmanager

def access_secret(name: str) -> str:
    client = secretmanager.SecretManagerServiceClient()
    project = os.environ["PROJECT_ID"]
    resp = client.access_secret_version(name=f"projects/{project}/secrets/{name}/versions/latest")
    return resp.payload.data.decode("utf-8")

NOTION_TOKEN = access_secret("notion-api-token")
NOTION_DB_ID = access_secret("notion-db-oportunidades-id")
HEADERS = {
    "Authorization": f"Bearer {NOTION_TOKEN}",
    "Notion-Version": "2022-06-28",
    "Content-Type": "application/json",
}

def create_notion_lead(request):
    envelope = request.get_json()
    if not envelope or "message" not in envelope:
        return ("Bad envelope", 400)

    msg = envelope["message"]
    lead = json.loads(base64.b64decode(msg["data"]).decode("utf-8"))

    # Dedup: chequear si el lead ya existe (mismo email)
    existe = requests.post(
        f"https://api.notion.com/v1/databases/{NOTION_DB_ID}/query",
        headers=HEADERS,
        json={"filter": {"property": "Email", "email": {"equals": lead["email"]}}}
    ).json()
    if existe.get("results"):
        return ("", 204)  # ya existe, ACK y salir

    requests.post(
        "https://api.notion.com/v1/pages",
        headers=HEADERS,
        json={
            "parent": {"database_id": NOTION_DB_ID},
            "properties": {
                "Nombre":  {"title": [{"text": {"content": f"{lead['nombre']} ({lead['empresa']})"}}]},
                "Empresa": {"rich_text": [{"text": {"content": lead["empresa"]}}]},
                "Email":   {"email": lead["email"]},
                "Estado":  {"select": {"name": "Nuevo"}},
                "Origen":  {"select": {"name": lead["origen"]}},
            }
        }
    )
    return ("", 204)  # ACK
```

```bash
# Crear topic con schema + DLQ y publicar de prueba
gcloud pubsub schemas create lead-schema --type=avro --definition-file=lead-schema.avsc
gcloud pubsub topics create innovaco.leads.nuevos --schema=lead-schema --message-encoding=json
gcloud pubsub topics create innovaco.leads.nuevos.dlq
gcloud pubsub topics publish innovaco.leads.nuevos \
  --message='{"nombre":"Juan","empresa":"Acme"}' --attribute=sector=formacion
```

### Trampas comunes
1. **No configurar DLQ**: los mensajes que fallan bloquean el consumo. Siempre DLQ + alerta.
2. **`ack_deadline` corto**: si tardas 5 min y el deadline es 60s, se reprocesa duplicado.
3. **Olvidar ack/nack**: en pull, sin ack el mensaje vuelve.
4. **Mensajes > 1 MB**: pasa el archivo a Cloud Storage y publica una **referencia**.
5. **No usar ordering keys** cuando importa el orden.
6. **Producir y consumir en el mismo servicio**: contraproducente.

### Ejercicio mental
> *"En horario laboral, los leads del sector 'pharma' notifican al chat de María; el resto al de Pedro; fuera de horario todos van a una database Notion 'Leads pendientes'. ¿Cómo lo modelas con un solo topic?"*

*Pista*: subscriptions filtradas por atributo `sector` + consumidor con lógica horaria, o un workflow disparado desde Pub/Sub.


---
## Bloque 4 — Eventarc

### Concepto en 30 segundos
**Enrutador de eventos** de GCP. Convierte cualquier evento (cambio en Cloud Storage, Audit Log de BigQuery, mensaje Pub/Sub, evento Firestore…) en una llamada a un destino: Cloud Run, Cloud Functions, Workflows, GKE.

> **Pub/Sub es la cola. Eventarc es el router** que decide quién consume qué.

### Anatomía
- **Trigger**: regla "qué evento dispara qué destino".
- **Event Source**: Cloud Storage, BigQuery, Cloud Run, Audit Logs, Pub/Sub, Firestore…
- **Filter**: condiciones (solo bucket X y archivos PDF).
- **Destination**: Cloud Run, Workflows, Functions, GKE, otro Pub/Sub.

### Cuándo cada herramienta

| Necesidad | Herramienta |
|---|---|
| "Que pase X cuando se sube un archivo a un bucket" | **Eventarc** |
| "Mantener una cola de eventos consumibles por varios" | **Pub/Sub** |
| "Sincronizar diariamente Notion / SaaS con BigQuery" | **Application Integration** |
| "Orquestar 8 pasos cuando llegue un contrato" | **Workflows** |

### Ejemplo concreto en InnovaCo
**Caso**: cuando se sube un PDF de propuesta firmada a `gs://innovaco-propuestas-firmadas/`: (1) se procesa con Document AI, (2) se inserta en `innovaco_raw_contratos.contratos_firmados`, (3) se dispara el workflow de onboarding del Bloque 2.

> **Nota didáctica**: usamos Cloud Storage como origen porque Eventarc tiene soporte nativo para eventos GCS, sin auth flow extra. En clase los alumnos suben un PDF con `gsutil cp ejemplo.pdf gs://...` y ven la cadena dispararse en tiempo real.


### Prompt para el LLM

```text
Eres un ingeniero senior de GCP especializado en arquitecturas event-driven.

CONTEXTO
Proyecto: innovaco-prod (region europe-west1).
Bucket existente: innovaco-propuestas-firmadas.
Workflow desplegado: onboarding-cliente-firmado (parámetro contrato_id).
Document AI processor: projects/innovaco-prod/locations/eu/processors/PROCESSOR_ID.
Tabla destino: innovaco_raw_contratos.contratos_firmados
  (contrato_id, cliente_id, cliente_nombre, valor NUMERIC, tipo_proyecto,
   fecha_subida TIMESTAMP, gcs_path STRING).

OBJETIVO
1. gcloud para crear un trigger Eventarc "propuesta-firmada-trigger" que:
   - se dispare con google.cloud.storage.object.v1.finalized,
   - SOLO del bucket innovaco-propuestas-firmadas,
   - SOLO archivos .pdf,
   - destino Cloud Run "procesar-propuesta-firmada" en europe-west1,
   - service account eventarc-sa@innovaco-prod.iam.gserviceaccount.com.
2. Cloud Run (Python Flask) que: parsee el CloudEvent, lea el archivo, lo procese con
   Document AI, extraiga campos, MERGE por contrato_id en BigQuery, y si es nuevo lance
   el workflow onboarding-cliente-firmado. 204 si OK, 500 con detalle si falla.

RESTRICCIONES
- Idempotencia: si Eventarc reenvía, no duplicar filas ni lanzar el workflow dos veces.
- Si Document AI confidence < 0.7: no insertar, loggear warning, devolver 204 (no reintentar).

FORMATO
1) gcloud (eventarc triggers create), 2) main.py, 3) requirements.txt.
```


### Lo que el LLM debería generar

```bash
# Eventarc trigger: archivo subido a Cloud Storage -> Cloud Run procesa
gcloud eventarc triggers create propuesta-firmada-trigger \
  --location=europe-west1 \
  --destination-run-service=procesar-propuesta-firmada \
  --destination-run-region=europe-west1 \
  --event-filters="type=google.cloud.storage.object.v1.finalized" \
  --event-filters="bucket=innovaco-propuestas-firmadas" \
  --service-account=eventarc-sa@innovaco-prod.iam.gserviceaccount.com
```

```python
@app.route("/", methods=["POST"])
def procesar():
    ce = from_http(request.headers, request.get_data())
    bucket = ce.data["bucket"]
    name = ce.data["name"]

    # 1. Document AI
    extracted = document_ai_client.process_document(f"gs://{bucket}/{name}")

    # 2. BigQuery raw con MERGE (idempotencia)
    bq_client.merge_rows(
        "innovaco_raw_contratos.contratos_firmados",
        rows=[extracted], merge_key="contrato_id")

    # 3. Lanzar workflow de onboarding (Bloque 2)
    workflows_client.execute(
        "onboarding-cliente-firmado", {"contrato_id": extracted["contrato_id"]})
    return "", 204
```

### Trampas comunes
1. **No usar filtros**: sin filter el trigger se dispara con todos los eventos del bucket. Caro y peligroso.
2. **Parsear el CloudEvent a mano**: usa el SDK `cloudevents`, no regex sobre el JSON.
3. **Olvidar idempotencia**: Eventarc es at-least-once.
4. **Eventos perdidos al cambiar de región**: Eventarc es regional; trigger y Cloud Run deben coincidir.

### Ejercicio mental
> *"¿Cómo detectas con Eventarc que un colaborador externo ha consultado datos de cliente en BigQuery, y avisas al DPO?"*

*Pista*: Audit Logs de BigQuery → Eventarc → filtro por usuario externo → Cloud Function que avisa por Telegram. Caso de **gobierno**: sin Eventarc sería un cronjob diario, no inmediato.


---
## Bloque 5 — Datastream

### Concepto en 30 segundos
Servicio **managed de Change Data Capture (CDC)**: replica cambios de una BD relacional (MySQL, Postgres, Oracle, SQL Server) a BigQuery o Cloud Storage **en near real-time**, sin tocar la app origen. Captura desde el binlog/WAL, no hace `SELECT *`. Es la versión serverless del viejo "ETL nocturno".

### Anatomía
- **Source**: BD origen + credenciales + tablas/columnas a replicar.
- **Destination**: BigQuery o Cloud Storage.
- **Stream**: la conexión activa que replica continuamente.
- **Backfill**: carga inicial histórica antes del CDC.
- **Private Connectivity**: VPC peering, Private Service Connect, IP allowlisting.

### Cuándo usarlo
- Hay una **BD relacional** importante y no quieres impactar su rendimiento.
- Necesitas datos en BigQuery **a los segundos**, no a la mañana siguiente.
- Quieres **bajar el acoplamiento** entre lo operacional y lo analítico.

### Cuándo NO usarlo
- **No tienes BD relacional** (InnovaCo en fase 0 no tiene → Datastream queda reservado).
- **BD NoSQL** (Mongo, DynamoDB): no soportado → Dataflow.
- **Volumen muy pequeño**: un `mysqldump` diario basta.

### Stack didáctico: Neon Postgres en lugar de Cloud SQL
Para hacerlo hands-on sin VPC/peering, usamos **Neon** (Postgres serverless gratis), con conectividad por **IP allowlisting** (más simple para clase). En producción real con BD propia: Cloud SQL Postgres con VPC privada.

**Caso didáctico**: el alumno crea una BD `tickets` en Neon, inserta 10 filas, y quiere que aparezcan en BigQuery y que cualquier cambio futuro se refleje en segundos.


### Prompt para el LLM

```text
Eres un ingeniero senior de GCP especializado en replicación de datos.

CONTEXTO
Proyecto: innovaco-prod (region europe-west1).
BD origen (stack didáctico): Neon Postgres serverless.
  - Host: ep-cool-xxx-1234.eu-central-1.aws.neon.tech, puerto 5432, db innovaco_tickets.
  - Usuario neon_admin (con REPLICATION). Password en Secret Manager "neon-password".
  - Conectividad pública con SSL. IP allowlisting (no VPC peering: Neon es serverless).
  - Tabla a replicar: public.tickets (ticket_id BIGSERIAL PK, cliente_id BIGINT, asunto TEXT,
    descripcion TEXT, estado VARCHAR, fecha_creacion TIMESTAMP, fecha_actualizacion TIMESTAMP).
Destino: BigQuery dataset innovaco_raw_postgres en europe-west1.

OBJETIVO
1. SQL en Neon: verificar wal_level=logical, crear slot "datastream_slot" (pgoutput),
   publication "datastream_publication" sobre public.tickets, permisos REPLICATION.
2. gcloud: connection profile source (Neon, IP allowlist), connection profile destination
   (BigQuery), stream "tickets-neon-a-bq" con backfill + CDC, y arrancarlo.
3. Query BigQuery de verificación (filas + max(fecha_actualizacion) para el lag).

RESTRICCIONES
- SSL obligatorio. Documentar que en prod real se usa VPC peering, no IP allowlist.
- Excluir columnas BLOB o > 1 MB. Alertar si lag de replicación > 60s.

FORMATO
Tres bloques: (1) SQL Postgres en Neon, (2) gcloud, (3) SQL de verificación BigQuery.
```


### Lo que el LLM debería generar

```sql
-- En la consola web de Neon o vía psql (db innovaco_tickets):
-- En Neon wal_level ya está en logical. En Cloud SQL / Postgres self-hosted:
--   ALTER SYSTEM SET wal_level = logical;  y reiniciar.

CREATE PUBLICATION datastream_publication FOR TABLE public.tickets;
SELECT pg_create_logical_replication_slot('datastream_slot', 'pgoutput');
ALTER ROLE neon_admin WITH REPLICATION;

-- (Opcional) usuario dedicado en lugar del admin:
-- CREATE ROLE datastream_user WITH LOGIN REPLICATION PASSWORD '...';
-- GRANT USAGE ON SCHEMA public TO datastream_user;
-- GRANT SELECT ON ALL TABLES IN SCHEMA public TO datastream_user;
```

```bash
# 1. Connection profile origen (Neon)
gcloud datastream connection-profiles create neon-source-profile \
  --location=europe-west1 --type=postgresql \
  --postgresql-hostname=ep-cool-xxx-1234.eu-central-1.aws.neon.tech \
  --postgresql-port=5432 --postgresql-database=innovaco_tickets \
  --postgresql-username=neon_admin \
  --postgresql-password="$(gcloud secrets versions access latest --secret=neon-password)" \
  --static-ip-connectivity   # Datastream usa IPs públicas conocidas -> allowlist en Neon

# 2. Connection profile destino (BigQuery)
gcloud datastream connection-profiles create bq-dest-profile \
  --location=europe-west1 --type=bigquery

# 3. Crear el stream (backfill + CDC continuo, solo public.tickets)
gcloud datastream streams create tickets-neon-a-bq \
  --location=europe-west1 --source=neon-source-profile \
  --postgresql-source-config='{"includeObjects":{"postgresqlSchemas":[{"schema":"public","postgresqlTables":[{"table":"tickets"}]}]},"publication":"datastream_publication","replicationSlot":"datastream_slot"}' \
  --destination=bq-dest-profile \
  --bigquery-destination-config='{"singleTargetDataset":{"datasetId":"innovaco-prod:innovaco_raw_postgres"}}' \
  --backfill-all --display-name="Tickets Neon -> BigQuery"

# 4. Arrancar
gcloud datastream streams update tickets-neon-a-bq \
  --location=europe-west1 --state=RUNNING --update-mask=state
```

```sql
-- Verificación en BigQuery
SELECT COUNT(*) AS filas, MAX(fecha_actualizacion) AS ultima_act
FROM `innovaco-prod.innovaco_raw_postgres.tickets`;
```

Cualquier `INSERT/UPDATE/DELETE` en `tickets` aparece en BigQuery en pocos segundos.

### Trampas comunes
1. **No habilitar logical replication / binlog** en la BD origen.
2. **Replicar columnas BLOB grandes**: dispara coste y latencia. Excluir.
3. **Olvidar que es eventual consistency**: hay segundos de retraso.
4. **No monitorizar el lag**: si crece silenciosamente, te enteras tarde. Alertas en el lag.

### Ejercicio mental
> *"Si InnovaCo migra HubSpot a un CRM propio en Postgres, ¿cómo evoluciona la ingesta? ¿Qué reemplaza Datastream y qué sigue igual?"*

*Pista*: el conector Application Integration ya no aplica; Datastream reemplaza la ingesta CRM. El resto del pipeline **no cambia** — es la promesa del embudo.


---
## Bloque 6 — Cloud Storage Transfer Service

### Concepto en 30 segundos
Servicio managed para **cargas batch grandes** desde fuentes externas a Cloud Storage o BigQuery: otros buckets cloud (S3, Azure Blob), FTP/SFTP, on-premise (vía agente), HTTPS. Pensado para mover **terabytes/petabytes**, con paralelización, reintentos, checksums y schedule.

### Anatomía
- **Job**: definición del traslado (origen, destino, filtros, schedule).
- **Agent**: binario que ejecutas en tu red para on-prem o sistemas privados.
- **Schedule**: una vez o recurrente. **Filters**: incluir/excluir por prefijo/fecha.

### Cuándo usarlo
- **Carga inicial masiva** (migrar 12.000 PDFs de Drive a GCS).
- **Sincronización entre clouds** (cliente entrega en S3).
- **Backups y archivado** a Coldline/Archive.
- **Cargas desde proveedores externos** (FTP → GCS).

### Cuándo NO usarlo
- **Streaming continuo** (es batch).
- **Pocos archivos pequeños** (basta `gsutil cp`).

### Ejemplo concreto en InnovaCo
**Caso**: un cliente comparte cada lunes una **carpeta Drive** con CSVs; los copiamos a `gs://innovaco-clientes-datos/cliente-x/` para que Dataflow los procese.

> **Nota didáctica**: Storage Transfer no tiene fuente nativa "Google Drive". En clase usamos el patrón **(A) Apps Script + Drive API + GCS API** (lo ejecutable). **(B)** Storage Transfer con SFTP se muestra como caso teórico de la herramienta "real".


### Prompt para el LLM (resumen)

```text
Eres un ingeniero senior de GCP.
CONTEXTO: carpeta Drive del cliente X (folder_id en Secret Manager), destino
gs://innovaco-clientes-datos/cliente-x/{año}/{semana}/ (Coldline a los 30 días),
trigger lunes 03:00 UTC, post-proceso por un Eventarc->Dataflow ya existente.

OBJETIVO — DOS variantes:
  VARIANTE A (clase): gcloud para bucket con lifecycle + SA, y Apps Script que cada lunes
    copia los CSV de Drive al bucket, idempotente (saltar si destino existe con mismo md5),
    y notifica por Pub/Sub "innovaco.transfers.estado".
  VARIANTE B (teórico): gcloud transfer jobs con fuente SFTP + agent pool + verificación checksum.
  VARIANTE C: query BigQuery sobre logs (tiempo, bytes, errores del último transfer).

RESTRICCIONES: cero credenciales en claro, idempotencia obligatoria.
FORMATO: tres bloques (A clase, B teórico, C verificación).
```


### Lo que el LLM debería generar

**Variante A — Apps Script Drive → GCS (lo que usamos en clase)**

```javascript
// drive_to_gcs.gs — corre cada lunes a las 03:00 UTC
function copyDriveCSVsToGCS() {
  const folderId = getSecret_("drive-clientex-folder-id");
  const bucket = "innovaco-clientes-datos";
  const prefix = `cliente-x/${currentYear_()}/${currentWeek_()}/`;
  const folder = DriveApp.getFolderById(folderId);
  const files = folder.getFilesByType(MimeType.CSV);
  let transferidos = 0, saltados = 0;

  while (files.hasNext()) {
    const file = files.next();
    const objectName = prefix + file.getName();
    if (gcsObjectExistsSameSize_(bucket, objectName, file.getSize())) { saltados++; continue; }
    uploadBlobToGCS_(bucket, objectName, file.getBlob());
    transferidos++;
  }

  publishStatus_("innovaco.transfers.estado", {
    cliente: "clientex", transferidos, saltados, ts: new Date().toISOString()
  });
}
// helpers gcsObjectExistsSameSize_, uploadBlobToGCS_, publishStatus_, getSecret_,
// currentYear_, currentWeek_ llaman a las APIs REST de GCS/Pub/Sub/Secret Manager
// con ScriptApp.getOAuthToken().
```

**Variante B — Caso teórico, Storage Transfer con SFTP**

```bash
gcloud transfer jobs create \
  posix:///tmp/transfer-config.json \
  gs://innovaco-clientes-datos/cliente-x/ \
  --description="Lunes - dump cliente X" \
  --schedule-starts="2026-05-25T03:00:00Z" \
  --schedule-repeats-every=7d \
  --source-creds-file=secret://sftp-creds
```

Tras la transferencia (ambas variantes), un Eventarc sobre el bucket dispara Dataflow. **La arquitectura del consumidor no cambia entre las dos variantes — esa es la lección.**

### Trampas comunes
1. **No validar permisos del origen** (el agente/credenciales deben leer).
2. **No usar Coldline para archivado**: Standard es 5x más caro para algo que no se lee.
3. **No habilitar checksums**: corrupción silenciosa = horas de debug.

### Ejercicio mental
> *"Comparado con `rsync` clásico, ¿qué te aporta Storage Transfer Service?"*

*Pista*: paralelización masiva, reintentos automáticos, schedule managed, logs centralizados, agente que cruza firewalls vía outbound HTTPS, escala a petabytes sin instalar nada.


---
## Bloque 7 — Cortex Framework

### Concepto en 30 segundos
**Blueprint open-source de Google** que entrega un **data foundation pre-construido** en BigQuery para fuentes SaaS comunes: SAP, Oracle EBS, Salesforce, Google Ads, Meta Ads, TikTok Ads, Google Workspace, ServiceNow… No es un producto: es **infraestructura como código** (Cloud Build + Dataform + scripts) que despliega conectores, esquemas canónicos, modelos analíticos, dashboards Looker y casos de uso de IA con Vertex AI.

### Cuándo usarlo
- Tu cliente tiene **al menos una** fuente Cortex.
- Quieres acelerar la fase 0 en **semanas** en vez de meses.
- Necesitas un **esquema canónico maduro** (más correcto que lo improvisado).

### Cuándo NO usarlo
- Tu cliente **no tiene** ninguna fuente soportada.
- Necesitas **total libertad de esquema** (Cortex impone su modelo).
- El equipo no domina dbt/Dataform.

### Decisión arquitectónica InnovaCo (ADR-004)
InnovaCo no tiene SAP ni Salesforce, pero **sí Google Workspace y Google Ads**. Por eso usa Cortex **parcialmente**: **SÍ** para Google Ads + Workspace; **NO** para HubSpot/Holded (no soportados → Application Integration custom).

### Anatomía del despliegue
1. Clone de `GoogleCloudPlatform/cortex-data-foundation`.
2. Configurar `config/config.json` (proyectos, datasets, fuentes activadas).
3. Cloud Build despliega: datasets `*_CDC` (crudo), `*_REPORTING` (analítico), DAGs/Workflows de refresco.
4. Dashboards Looker opcionales.


### Prompt para el LLM

```text
Eres un ingeniero senior de GCP especializado en data foundations.

CONTEXTO
InnovaCo despliega Cortex Framework parcialmente.
  - innovaco-source (datasets brutos de los conectores).
  - innovaco-prod (reporting y modelos analíticos).
Region BigQuery: EU. Moneda: EUR. Idiomas: es, en.
Fuentes a activar:
  - Google Ads (MCC 123-456-7890, customer IDs 111111, 222222).
  - Google Workspace: Gmail, Drive, Calendar (SA con domain-wide delegation).
Fuentes a NO activar: Meta Ads, TikTok, CM360, SAP, Salesforce, Oracle EBS.
Cross-media (k9.deployCrossMedia): false.
Repo: github.com/GoogleCloudPlatform/cortex-data-foundation (tag estable pinned, NO main).

OBJETIVO
1. config/config.json completo, con false explícito en lo que no se usa.
2. gcloud previos: habilitar APIs, bucket "innovaco-cortex" (EU), roles a la SA de Cloud Build.
3. gcloud builds submit con substitutions _PJID_SRC y _PJID_TGT.
4. Query BigQuery de validación: K9_PROCESSING, GoogleAds_REPORTING, Workspace_REPORTING.

RESTRICCIONES
- Versionar config.json con el tag pinned de Cortex.
- No modificar los modelos de Cortex; derivados en innovaco-prod.custom_analytics.

FORMATO
Cuatro bloques: (1) config.json, (2) gcloud previos, (3) despliegue, (4) SQL de validación.
```


### Lo que el LLM debería generar

```json
// config/config.json (extracto)
{
  "projectIdSource": "innovaco-source",
  "projectIdTarget": "innovaco-prod",
  "targetBucket": "innovaco-cortex",
  "location": "EU",
  "currencies": ["EUR"],
  "languages": ["es", "en"],
  "deployMarketing": {
    "deployGoogleAds": true,
    "deployCM360": false,
    "deployTikTok": false,
    "deployMeta": false
  },
  "deployWorkspace": {
    "deployGmail": true,
    "deployDrive": true,
    "deployCalendar": true
  },
  "k9": { "deployCrossMedia": false }
}
```

```bash
# Despliegue
gcloud builds submit \
  --config cloudbuild.deploy.yaml \
  --substitutions=_PJID_SRC=innovaco-source,_PJID_TGT=innovaco-prod
```

Tras ~45 min aparecen en BigQuery: `K9_PROCESSING` (campos canónicos cross-fuente),
`GoogleAds_REPORTING` (CPC, conversiones, atribución), `Workspace_REPORTING` (uso de
Drive/Gmail/Calendar), y los dashboards Looker listos para conectar.

### Trampas comunes
1. **Desplegar todo "por si acaso"**: enciendes 8 fuentes que no tienes → datasets vacíos.
2. **No fijar la versión del repo**: clonar `main` arriesga breaking changes.
3. **Modificar los modelos de Cortex**: rompes la capacidad de actualizar. Usa datasets derivados.
4. **Esperar ETL de SaaS no soportados**: entiende qué cubre y qué no.

### Ejercicio mental
> *"InnovaCo compra un competidor que usa Salesforce. ¿Qué cambia en tu data foundation? ¿Migras HubSpot o mantienes ambos?"*

*Pista*: con Salesforce, Cortex pasa de "parcial" a "principal". Mantener dos CRMs raramente se justifica. Es un ADR.


---
## Bloque 8 — Cierre y decisión arquitectónica integradora

### Recapitulación — qué herramienta usar para qué

```
┌─────────────────────────────────────────────────────────────┐
│  ¿Qué quieres ingestar?                                      │
├─────────────────────────────────────────────────────────────┤
│  SaaS con conector pre-hecho     →  Application Integration  │
│  SaaS no soportado / lógica       →  Application Integration  │
│  compleja                            + REST custom            │
│  BD relacional con CDC           →  Datastream               │
│  Fuente batch grande / cliente   →  Storage Transfer Service │
│  externo                                                     │
│  Eventos GCP (archivos, logs)    →  Eventarc                 │
│  Eventos de negocio internos     →  Pub/Sub                  │
│  Orquestación multi-paso         →  Workflows                │
│  Stack canónico (SAP, SF, Ads…)  →  Cortex Framework         │
└─────────────────────────────────────────────────────────────┘
```

### La estrategia de InnovaCo en una frase

> **Cortex para lo canónico, Application Integration para lo SaaS, Workflows para la orquestación, Pub/Sub como sistema nervioso, Eventarc como sensor, Datastream y Storage Transfer reservados para cuando aplique.**

### Conexión con el resto del embudo
Lo ingerido aquí cae en BigQuery (capa 2), se enriquece (capa 3), se vectoriza (capa 4), lo razona la IA (capa 5) y se entrega como output (capa 6). **Toda la potencia AI-first nace de lo que decidamos meter en este embudo.**

### Pregunta final (entrega)
> *"Toma una empresa real que conozcas. Dibuja qué fuente va por cada una de las 7 tecnologías. Las que no encajen en ninguna, márcalas — ahí está tu siguiente conversación con el CTO."*


In [ ]:
# Recomendador de tecnología — mini-juego del cierre. Ejecútala y prueba tus casos.
REGLAS = [
    (("saas", "conector", "hubspot", "salesforce", "slack", "notion"), "Application Integration"),
    (("bd", "postgres", "mysql", "cdc", "replica", "base de datos"),     "Datastream"),
    (("batch", "masivo", "terabyte", "sftp", "s3", "migrar", "backup"),  "Storage Transfer Service"),
    (("archivo", "bucket", "subido", "audit log", "evento gcp", "pdf"),  "Eventarc"),
    (("evento", "fan-out", "cola", "desacoplar", "lead", "mensaje"),     "Pub/Sub"),
    (("orquestar", "pasos", "onboarding", "condicional", "paralelo"),    "Workflows"),
    (("canonico", "sap", "google ads", "workspace", "kpi", "dashboard"), "Cortex Framework"),
]

def recomienda_tecnologia(caso: str) -> str:
    c = caso.lower()
    hits = [(tec, sum(k in c for k in keys)) for keys, tec in REGLAS]
    hits = [(tec, n) for tec, n in hits if n > 0]
    if not hits:
        return "Sin match claro: replantea el caso o combina varias tecnologías."
    hits.sort(key=lambda x: x[1], reverse=True)
    top = hits[0][1]
    elegidas = [tec for tec, n in hits if n == top]
    return " / ".join(elegidas)

ejemplos = [
    "Sincronizar diariamente la base Notion de oportunidades con BigQuery",
    "Replicar una base de datos Postgres a BigQuery en segundos",
    "Procesar un PDF en cuanto se sube a un bucket de Cloud Storage",
    "Hacer fan-out de un lead nuevo a tres consumidores independientes",
    "Orquestar el onboarding de un cliente en 4 pasos paralelos",
    "Desplegar KPIs canónicos de Google Ads y Workspace",
    "Migrar 12.000 PDFs históricos desde un SFTP del cliente",
]
for caso in ejemplos:
    print(f"- {caso}\n  -> {recomienda_tecnologia(caso)}\n")


---
## Apéndice A — Comandos gcloud útiles

```bash
# Application Integration
gcloud integration integrations list --location=europe-west1
gcloud integration executions describe EXEC_ID --integration=NAME --location=europe-west1

# Workflows
gcloud workflows deploy NAME --source=workflow.yaml --location=europe-west1
gcloud workflows execute NAME --data='{"key":"value"}' --location=europe-west1

# Pub/Sub
gcloud pubsub topics create TOPIC
gcloud pubsub subscriptions create SUB --topic=TOPIC --push-endpoint=URL
gcloud pubsub topics publish TOPIC --message='{"k":"v"}' --attribute=key=value

# Eventarc
gcloud eventarc triggers create NAME --location=europe-west1 \
  --destination-run-service=SERVICE \
  --event-filters="type=google.cloud.storage.object.v1.finalized" \
  --event-filters="bucket=BUCKET"

# Datastream
gcloud datastream streams list --location=europe-west1

# Storage Transfer
gcloud transfer jobs create SOURCE DESTINATION --description="..."

# Cortex (despliegue)
git clone https://github.com/GoogleCloudPlatform/cortex-data-foundation.git
gcloud builds submit --config cloudbuild.deploy.yaml
```

## Apéndice B — Recursos
- Application Integration: cloud.google.com/application-integration/docs
- Workflows: cloud.google.com/workflows/docs
- Pub/Sub: cloud.google.com/pubsub/docs
- Eventarc: cloud.google.com/eventarc/docs
- Datastream: cloud.google.com/datastream/docs
- Storage Transfer: cloud.google.com/storage-transfer/docs
- Cortex Framework: cloud.google.com/cortex/docs
- Repo Cortex: github.com/GoogleCloudPlatform/cortex-data-foundation


## Apéndice C — Trampas que se repiten en TODAS las tecnologías

Las trampas son **patrones**, no productos. Quien aprende estos evita el 80% de los problemas:

1. **Idempotencia obligatoria** — merge en vez de insert, upsert en vez de append, idempotency keys.
2. **Logging y trazabilidad** — responder "¿qué pasó con ese lead del 15 de mayo?" sin un dump de la BD.
3. **Service accounts dedicadas y mínimas** — una SA por flujo, con roles custom.
4. **Secret Manager para todo lo sensible** — nunca constantes ni env vars hardcodeadas.
5. **DLQ siempre** — los mensajes que fallan van a cola muerta + alerta, no se pierden.
6. **Monitorización de lag** — para CDC, batch y streaming.
7. **Versionado de configuración** — YAML, JSON, schemas, repos Cortex en Git.
8. **Validación de schema** — validar contra Avro/Protobuf antes de escribir en BigQuery.


## Apéndice D — Biblioteca de prompts (plantilla maestra y convenciones)

Los 7 prompts de la clase comparten estructura. Esta es la **plantilla maestra** para crear nuevos:

```text
Eres un ingeniero senior de Google Cloud Platform.

CONTEXTO
[Proyecto, región, service accounts, conexiones existentes, secretos en Secret Manager]

DATOS DE ENTRADA / ESQUEMA
[Tablas BigQuery, columnas, tipos, parámetros de entrada esperados]

OBJETIVO
[Qué producir: YAML, script, gcloud command. Numerar pasos]

RESTRICCIONES
- Idempotencia: ...
- Cero secretos en claro: Secret Manager
- Retries con backoff exponencial
- Logging a Cloud Logging
- Service account: ...
- Sin hardcodear IDs/región

FORMATO
[Cómo devolver: bloques separados, solo código sin explicación, comentarios sí/no]
```

**Convenciones constantes** (lo que convierte los prompts en plantillas reutilizables):
- **Proyecto**: `innovaco-prod` (o `-dev` / `-stg`).
- **Región**: `europe-west1`.
- **Naming**: topics `innovaco.<dominio>.<evento>`; datasets `innovaco_<zona>_<dominio>`;
  service accounts `<funcion>-sa@innovaco-prod.iam.gserviceaccount.com`;
  workflows `<verbo>-<sustantivo>` kebab-case.
- **Secretos**: siempre en Secret Manager, referenciados por nombre.

> **Revisa el output antes de desplegar.** La IA no sabe si tu bucket existe, si la SA tiene permisos o si el secreto está creado. Tú sí. La revisión humana es el control del nivel N1 de autonomía.


In [ ]:
# Plantilla maestra de prompt, parametrizable — "los prompts son código".
PLANTILLA_MAESTRA = """Eres un ingeniero senior de Google Cloud Platform.

CONTEXTO
Proyecto: {proyecto} (region {region}).
Service account: {service_account}.
Secretos en Secret Manager: {secretos}.

DATOS DE ENTRADA / ESQUEMA
{esquema}

OBJETIVO
{objetivo}

RESTRICCIONES
- Idempotencia (merge/upsert por clave).
- Cero secretos en claro: Secret Manager.
- Retries con backoff exponencial.
- Logging a Cloud Logging.
- Sin hardcodear IDs ni región.

FORMATO
{formato}
"""

prompt = PLANTILLA_MAESTRA.format(
    proyecto="innovaco-prod",
    region="europe-west1",
    service_account="integration-notion-sa@innovaco-prod.iam.gserviceaccount.com",
    secretos="notion-api-token, notion-db-oportunidades-id",
    esquema="Notion 'Oportunidades' -> BigQuery innovaco_raw_notion.oportunidades (MERGE por oportunidad_id).",
    objetivo="Genera una integración que sincronice cada día a las 03:00 Europe/Madrid.",
    formato="Devuélveme solo el YAML, sin explicación.",
)
print(prompt)


## Apéndice E — Setup del stack didáctico en 30 minutos

Guía para preparar **antes** de la clase. Si llegas con el stack creado, los 90 min rinden el doble.

**Paso 1 — Notion (5 min):** cuenta gratis → workspace "InnovaCo-clase" → 3 databases
(`Oportunidades`, `Clientes`, `Proyectos`) → crear integración interna "InnovaCo Clase"
(copiar token `secret_...`) → compartir las 3 databases con la integración → anotar los database IDs.

**Paso 2 — Telegram bot (3 min):** `@BotFather` → `/newbot` → copiar token → crear grupo,
añadir el bot, mandar un mensaje → obtener `chat_id` desde
`https://api.telegram.org/bot{TOKEN}/getUpdates` (empieza por `-100` para grupos).

**Paso 3 — Google Workspace (5 min):** Drive/Forms/Sheets/Calendar activos → carpeta raíz
"InnovaCo - Clientes" (anotar folder ID) → Form "Lead nuevo" (Nombre, Empresa, Email, Interés,
Origen) con respuestas en un Sheet → Sheet "clientes-maestro" hoja "Clientes".

**Paso 4 — Google Cloud (10 min):** proyecto `innovaco-prod`, billing, habilitar APIs:

```bash
gcloud services enable \
  bigquery.googleapis.com storage.googleapis.com pubsub.googleapis.com \
  eventarc.googleapis.com workflows.googleapis.com cloudfunctions.googleapis.com \
  run.googleapis.com secretmanager.googleapis.com datastream.googleapis.com \
  documentai.googleapis.com integrations.googleapis.com --project=innovaco-prod
```

Crear secretos, datasets y bucket:

```bash
echo -n "secret_xxxxxxx" | gcloud secrets create notion-api-token --data-file=-
echo -n "DB_ID_OPORTUNIDADES" | gcloud secrets create notion-db-oportunidades-id --data-file=-
echo -n "123456:ABC..." | gcloud secrets create telegram-bot-token --data-file=-
echo -n "-100xxxxx" | gcloud secrets create telegram-chat-id --data-file=-
# ... (resto de IDs Notion/Drive/Sheets)

bq mk --location=EU innovaco_raw_notion
bq mk --location=EU innovaco_curated_contratos
bq mk --location=EU innovaco_raw_postgres

gcloud storage buckets create gs://innovaco-propuestas-firmadas \
  --location=EU --uniform-bucket-level-access
```

**Paso 5 — Neon Postgres (2 min, opcional, solo Bloque 5):** cuenta neon.tech → proyecto
"innovaco-tickets" (región EU) → crear tabla `tickets` con 2 filas → password en
`gcloud secrets create neon-password`.

**Coste de la clase:** todo entra en **free tier** → **0 € durante la sesión**.


In [ ]:
# Checklist de setup — márcala antes de empezar la clase.
checklist = [
    "Notion: 3 databases creadas y compartidas con la integración",
    "Token Notion + database IDs en Secret Manager",
    "Bot Telegram creado, en un grupo, mensaje de prueba enviado",
    "Token Telegram + chat_id en Secret Manager",
    "Workspace: carpeta Drive raíz + Form + Sheet conectados",
    "IDs Drive y Sheets en Secret Manager",
    "Proyecto GCP creado, APIs habilitadas",
    "BigQuery datasets creados, bucket creado",
    "(Opcional) Neon: project + tabla tickets con 2 filas",
]
print("Checklist de preparación del stack didáctico:\n")
for item in checklist:
    print(f"  [ ] {item}")
print("\nCuando todo esté en verde, pasa al Notebook 0 — Fundamentos y setup.")


---

*Clase v2.0 — basada en el material de InnovaCo (stack didáctico Notion + Workspace + Telegram + Neon).*
*Siguiente: **Notebook 0 — Fundamentos y setup**, donde montas el entorno y ejecutas tu primer comando real en GCP.*
